In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when,to_timestamp
from pyspark.sql.types import StructType, StructField, StringType



In [ ]:
spark = SparkSession.builder \
    .appName("MACCDCDataAnalysis") \
    .getOrCreate()

import os
base_path = os.environ.get("MACCDC_DATA_DIR", "data/maccdc-2012")
folders = ["00", "01", "02", "03", "04", "05"]
http_files = [f"{base_path}/{folder}/http.log.gz" for folder in folders]
dns_files = [f"{base_path}/{folder}/dns.log.gz" for folder in folders]

http_df = spark.read.option("header", "true").json(http_files)
dns_df = spark.read.option("header", "true").json(dns_files)

# Convert the 'timestamp' column to a timestamp type
http_df = http_df.withColumn("ts", to_timestamp(col("ts")))
dns_df = dns_df.withColumn("ts", to_timestamp(col("ts")))

# Name http_log and dns_log
http_df.createOrReplaceTempView("http_log")
dns_df.createOrReplaceTempView("dns_log")

In [ ]:
http_df.show(5)

In [ ]:
dns_df.show(5)

In [ ]:
result_sql = spark.sql("""
SELECT uri, COUNT(*) AS access_count
FROM http_log
WHERE status_code = '200' AND method = 'GET'
GROUP BY uri
ORDER BY access_count DESC
""")
result_sql.createTempView("http_access_count_table")


In [ ]:
result_sql.show()

In [ ]:
result_join = spark.sql("""
SELECT 
  *
FROM http_log AS h
JOIN dns_log AS d
  ON h.uid = d.uid
""")
result_join.show()


In [ ]:
tcp_percentage = spark.sql("""
SELECT
    uri, tcp_percentage
FROM (
    SELECT
        uri,
        COUNT(CASE WHEN dns_log.proto = 'TCP' THEN 1 END) * 100.0 / COUNT(*) AS tcp_percentage
    FROM http_log
    JOIN dns_log ON http_log.uid = dns_log.uid
    WHERE http_log.status_code = '200' AND http_log.method = 'GET'
    GROUP BY uri
) AS tcp_counts
WHERE tcp_percentage > 50
""")

tcp_percentage.createTempView("tcp_percentage_table")
tcp_percentage.show()